# Rigid Pendulum on SO(3)

Derives the equations of motion for a rigid body rotating about a fixed pivot
under gravity, using the geometric formulation from Lee, Leok, McClamroch (2018),
Chapter 6.

**Configuration manifold**: $SO(3) = \{R \in \mathbb{R}^{3\times3} : R^TR = I,\, \det R = 1\}$

**Kinematics**: $\dot{R} = R\widehat{\Omega}$ where $\Omega$ is the body-frame angular velocity

**EOM** (Lee et al. Eq 6.8):
$$J\dot{\Omega} + \Omega \times J\Omega + mg\,\rho \times R^T e_3 = M$$

In [1]:
# Install from GitHub (for Colab); uncomment if not installed locally
# !pip install -q git+https://github.com/vkotaru/pygeomech.git@geomech

from geomech import (
    SO3, Scalar, Vector, Matrix, Dot, Cross,
    SystemVariables, TimeDerivative,
    compute_eom, to_standard_form, getScalars,
    to_latex, display_eom, display_standard_form,
)
from geomech.core.operations.multiplication import MVMul
from geomech.utils.printing import print_tree
from IPython.display import Math

## 1. System definition

A rigid body of mass $m$ rotates about a fixed pivot. The center of mass
is located at $\rho$ in the body frame relative to the pivot. $J$ is the
inertia matrix about the pivot (includes parallel axis contribution).

In [2]:
# Parameters
m, g = getScalars('m g', attr=['Constant'])
J = Matrix('J', attr=['Constant', 'SymmetricMatrix'])  # inertia about pivot
rho = Vector('\\rho', attr=['Constant'])  # pivot to COM in body frame
e3 = Vector('e3', attr=['Constant'])       # gravity direction
half = Scalar('0.5', value=0.5, attr=['Constant'])

# Attitude
R = SO3('R')
Om = R.get_tangent_vector()    # body angular velocity Ω
eta = R.get_variation_vector()  # variation vector η

# Input: external torque
M_torque = Vector('M')

print('R ∈ SO(3)')
print('Ω =', Om)
print('η =', eta)

R ∈ SO(3)
Ω = \Omega_{R}
η = \eta_{R}


## 2. Lagrangian

$$L = \frac{1}{2} \Omega^T J \Omega - mg\, (R\rho) \cdot e_3$$

The kinetic energy uses $J$ about the pivot. The potential energy is
$mg$ times the height of the COM: $(R\rho) \cdot e_3$.

In [3]:
KE = Dot(Om, J * Om) * half
PE = m * g * Dot(MVMul(R, rho), e3)
L = KE - PE

display(Math(r'KE = ' + to_latex(KE)))
display(Math(r'PE = ' + to_latex(PE)))
display(Math(r'L = ' + to_latex(L)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Infinitesimal work

$$\delta W = \eta \cdot M$$

In [4]:
dW = Dot(eta, M_torque)
display(Math(r'\delta W = ' + to_latex(dW)))

<IPython.core.display.Math object>

## 4. Equations of motion

In [5]:
variables = SystemVariables(matrices=[R])
eom = compute_eom(L, dW, variables)

key = list(eom.keys())[0]
_, eqn = eom[key]

display_eom(eom)

<IPython.core.display.Math object>

## 5. Expression tree

In [6]:
print_tree(eqn)

                                                                 VAdd
               /                                        /                                        \                      \
              S*V                                      S*V                                      S*V                    v:M
       /                \                   /                       \               /                         \
      M*V             -1.0*               Cross                    -1*            Cross                      Mul
  /          \                      /               \                       /               \             /       \
M:J*       d/dt               v:\Omega_{R}         M*V                   v:\rho*           M*V           Mul     -1*
             |                                 /          \                            /         \      /    \
       v:\Omega_{R}                          M:J*   v:\Omega_{R}                   Transpose   v:e3*   m*   g*
     

## 6. Standard form

$$M(q)\dot{\Omega} + f(q, \Omega) + G\,u = 0$$

In [7]:
sf = to_standard_form(eom, variables, [M_torque])
display_standard_form(sf)

<IPython.core.display.Math object>

## 7. Verify against Lee et al. Eq 6.8

Expected:
$$J\dot{\Omega} + \Omega \times J\Omega + mg\,\rho \times R^Te_3 = M$$

In [8]:
eq = sf[key]
ddOm = str(TimeDerivative(Om))

display(Math(r'M[\dot{\Omega}] = ' + to_latex(eq.M[ddOm])))
display(Math(r'f = ' + to_latex(eq.f)))
display(Math(r'G[M] = ' + to_latex(eq.G['M'])))
print()
print('J*Ω̇ + Ω×JΩ + mg*ρ×(R^T e3) = M  ✓')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


J*Ω̇ + Ω×JΩ + mg*ρ×(R^T e3) = M  ✓
